# **Initialize the Model (a100)**

In [ ]:
import sys

# Uninstall potentially conflicting packages to ensure a clean slate
!pip uninstall -y vllm triton torch torchvision torchaudio Pillow

# Install torch and torchvision specifically for CUDA 12.x (common in Colab)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install vllm allowing it to pick a compatible version, with transformers pinned to 4.44.2
!pip install vllm==0.6.0 transformers==4.44.2 --quiet

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 106.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 115.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

def get_secret(key_name):
    """Fetch a secret from Colab, Kaggle, or env — whichever is available."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
    return os.getenv(key_name)

hf_token = get_secret("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — add it in Kaggle > Add-ons > Secrets"

# Log in so vLLM can pull the gated model
from huggingface_hub import login
login(token=hf_token)

In [ ]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"

llm = LLM(
    model=MODEL_ID,
    quantization="awq_marlin",     # awq_marlin = faster fused kernels; falls back to awq if unsupported
    dtype="half",                   # fp16 — correct for AWQ
    gpu_memory_utilization=0.90,    # leave 10% headroom for KV cache spikes
    max_model_len=8192,             # FIXED: cap context to save VRAM; raise toward 16384 if you have headroom
    tensor_parallel_size=1,         # single GPU (A100 in Colab is 1 × 40GB)
    trust_remote_code=True,         # Qwen needs this
)

tokenizer = llm.get_tokenizer()
print("Model loaded ✓")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 06-18 17:12:28 awq_marlin.py:89] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-18 17:12:28 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post2) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, serv

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

INFO 06-18 17:12:31 model_runner.py:997] Starting to load model Qwen/Qwen2.5-32B-Instruct-AWQ...


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:230: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# **G4 Dependencies**

In [ ]:
# ── Blackwell GPU (sm_120: RTX PRO 6000, RTX 5090, GB200, etc.) ─────────────
# Root cause of failures:
#   • PyTorch built for cu121 only supports up to sm_90 (Ada/Hopper).
#   • vLLM 0.6.x predates Blackwell support.
# Fix: CUDA 12.8 wheel index + PyTorch 2.7+ + vLLM 0.8+

!pip uninstall -y vllm triton torch torchvision torchaudio -q

# PyTorch 2.7+ with CUDA 12.8 — first release to include sm_120 kernels
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --quiet

# vLLM 0.8+ adds Blackwell (sm_120) support; transformers 4.45+ required by vLLM 0.8
# Pinning to vLLM 0.8.0 to ensure CUDA 12.x compatibility.
!pip install "vllm==0.8.0" "transformers>=4.45.0" --quiet

print("✓ Blackwell dependencies installed (cu128 / PyTorch 2.7+ / vLLM 0.8+)")

# Quick sanity check
import torch
print(f"  torch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f"  GPU: {torch.cuda.get_device_name(0)}  (sm_{cap[0]}{cap[1]})")
    assert cap >= (12, 0), f"Expected sm_120+, got sm_{cap[0]}{cap[1]}"
    print("  ✓ sm_120 confirmed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.3/265.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


# **Generate Prompts**

## **Generate Instructions for the Prompts**

In [ ]:
"""
build_query_instructions.py
============================
STAGE 1 of the query-generation pipeline.

This script does ONE thing: it builds the *instruction prompts* (the meta-prompts
you feed to the big model, e.g. Qwen2.5-32B) that will make it emit QUERY TEMPLATES.
It does not call any model and it does not generate the final queries.

The flow it sets up:

    [this script]            build_instructions()  -> HF dataset of instruction prompts
    [stage 2, your notebook] feed each instruction -> model -> list of query TEMPLATES
    [stage 3, this script]   expand_templates()    -> concrete queries (idx 0-22, groups, etc.)

A query TEMPLATE is a question with literal placeholders the model is told to embed,
e.g.  "Sensor {idx}, are you picking up any {modality_phrase}?"
We expand those placeholders locally (the "x23" you wanted) instead of paying for a
model call per sensor. See PLACEHOLDERS and expand_templates() at the bottom.

Run:  python build_query_instructions.py
Out:  ./query_instructions_ds/   (datasets.save_to_disk)  +  ./query_instructions.jsonl
"""

import json
import random
import itertools
from datasets import Dataset

random.seed(0)

# ---------------------------------------------------------------------------
# 1. WORLD MODEL  — injected into every instruction so the model writes
#    faithful, answerable questions. Kept tight on purpose (token budget).
# ---------------------------------------------------------------------------
WORLD_MODEL = """\
SYSTEM UNDER OBSERVATION — airport counter-drone network:
- 23 fixed sensors, indexed 0-22, arranged in a ring around the airfield.
- Each sensor carries three independent detectors:
    * RF      — classifies radio emissions as FRIENDLY or THREAT.
    * Audio   — classifies sound as MAMBO drone (threat), BEBOP drone (threat),
                or BACKGROUND noise (friendly).
    * Visual  — a camera score from 0 (no drone) to 1 (drone present).
- A fusion model combines everything into a single system-wide threat confidence.
- Named sensor groups an operator may refer to:
    * Quadrants:  Q1 = sensors 11-16,  Q2 = 5-11,  Q3 = 0-5,  Q4 = 16-22.
    * Hemispheres: North = 5-16,  South = 0-5 & 16-22,
                   East = 11-22,  West = 0-11.
Every question must be answerable from a single sensor snapshot. Questions never
state specific readings or numbers — they ASK for them."""

# ---------------------------------------------------------------------------
# 2. PERSONAS — name + register hint so the phrasing varies by speaker.
# ---------------------------------------------------------------------------
PERSONAS = {
    "formal_commander":        "a formal military commander; clipped, directive, uses chain-of-command phrasing",
    "security_analyst":        "a security analyst; precise, data-oriented, asks about confidence and evidence",
    "air_traffic_controller":  "an air traffic controller; brisk, airspace-focused, plain radio cadence",
    "federal_air_marshal":     "a federal air marshal; security-minded, asks about intent and severity",
    "airport_ops_officer":     "an airport operations officer; logistics-minded, asks about impact and zones",
    "first_responder":         "an on-scene first responder; urgent, action-oriented, short questions",
    "incident_commander":      "an incident commander; coordinating, asks for status and prioritization",
}

# ---------------------------------------------------------------------------
# 3. MODALITIES — the {modality_phrase} slot expands from natural phrasings,
#    NOT the bare word. This is the answer to "can modality be an f-string?":
#    yes, but fill it from here so queries read naturally.
# ---------------------------------------------------------------------------
MODALITY_PHRASES = {
    "rf":      ["RF transmissions", "the RF signature", "radio-frequency activity"],
    "audio":   ["the acoustic signature", "drone audio (mambo/bebop)", "the audio detector"],
    "visual":  ["the camera feed", "visual confirmation", "the optical detector"],
    "all":     ["any detector", "RF, audio, or visual", "any of its three sensors"],
}

# Named groups -> index sets, exactly as you defined them (index-based, verbatim).
GROUPS = {
    "the first quadrant":     list(range(11, 17)),
    "the second quadrant":    list(range(5, 12)),
    "the third quadrant":     list(range(0, 6)),
    "the fourth quadrant":    list(range(16, 23)),
    "the northern hemisphere": list(range(5, 17)),
    "the southern hemisphere": list(range(0, 6)) + list(range(16, 23)),
    "the eastern hemisphere":  list(range(11, 23)),
    "the western hemisphere":  list(range(0, 12)),
}

# ---------------------------------------------------------------------------
# 4. PLACEHOLDERS — tokens the model is told to embed verbatim in its query
#    templates, and how stage 3 expands each. (Expansion lives at the bottom.)
# ---------------------------------------------------------------------------
PLACEHOLDERS = {
    "{idx}":            "a single sensor index 0-22",
    "{group}":          "a named sensor group, e.g. 'the northern hemisphere'",
    "{indices}":        "an explicit list of sensor indices, e.g. '2, 6, 14'",
    "{modality_phrase}": "a detector phrase, e.g. 'the acoustic signature'",
}

# ---------------------------------------------------------------------------
# 5. TASK SPECS — each entry defines one *kind* of instruction. `directive`
#    is what we tell the model to produce; `placeholders` is what it must embed;
#    `modality_scopes` controls whether we also fan out over modalities.
# ---------------------------------------------------------------------------
TASKS = {
    "overall_threat": dict(
        placeholders=[],
        modality_scopes=[None],
        directive=("questions that ask whether there is a threat ANYWHERE in the "
                   "system right now — a single overall yes/no/elaborate judgment."),
    ),
    "single_sensor": dict(
        placeholders=["{idx}", "{modality_phrase}"],
        modality_scopes=["rf", "audio", "visual", "all"],
        directive=("questions interrogating ONE sensor, written with the literal "
                   "placeholder {idx} where the sensor number goes and "
                   "{modality_phrase} where the detector goes. Ask it to report "
                   "confidence or make a threat call for that sensor."),
    ),
    "check_all": dict(
        placeholders=["{modality_phrase}"],
        modality_scopes=["rf", "audio", "visual", "all"],
        directive=("questions that sweep ALL 23 sensors at once for "
                   "{modality_phrase} — e.g. flag every sensor that sees something."),
    ),
    "multi_sensor_group": dict(
        placeholders=["{group}", "{modality_phrase}"],
        modality_scopes=["rf", "audio", "visual", "all"],
        directive=("questions about a NAMED group of sensors, using the literal "
                   "placeholder {group} for the group name and {modality_phrase} "
                   "for the detector. Ask for a group-level status or threat call."),
    ),
    "multi_sensor_list": dict(
        placeholders=["{indices}", "{modality_phrase}"],
        modality_scopes=["rf", "audio", "visual", "all"],
        directive=("questions about an arbitrary handful of sensors, using the "
                   "literal placeholder {indices} for the list (e.g. '2, 6, 14') "
                   "and {modality_phrase} for the detector."),
    ),
    "ranking": dict(
        placeholders=[],
        modality_scopes=[None],
        directive=("questions that ask which sensor (or sensors) is MOST alarmed / "
                   "highest threat confidence — a ranking or single most-likely call."),
    ),
    "tasking": dict(
        placeholders=[],
        modality_scopes=[None],
        directive=("questions asking what ACTION to take given the situation — how "
                   "severe it is, whether to dispatch or respond, how urgent."),
    ),
}

# ---------------------------------------------------------------------------
# 6. Few-shot style anchors (rotated, not fixed, to avoid mode collapse).
#    Style only — the model should not copy them verbatim.
# ---------------------------------------------------------------------------
STYLE_EXAMPLES = {
    "single_sensor": [
        "Sensor {idx} — what is {modality_phrase} telling you?",
        "Give me {idx}'s read on {modality_phrase}.",
        "Is sensor {idx} flagging anything on {modality_phrase}?",
    ],
    "multi_sensor_group": [
        "Status on {group} — anything on {modality_phrase}?",
        "Does {group} show a threat across {modality_phrase}?",
    ],
    "multi_sensor_list": [
        "Check sensors {indices} for {modality_phrase}.",
        "What do {indices} report on {modality_phrase}?",
    ],
    "check_all": [
        "Sweep every sensor — who's seeing {modality_phrase}?",
        "Any sensor at all picking up {modality_phrase}?",
    ],
    "overall_threat": [
        "Do we have a threat in the airspace right now?",
        "Bottom line — is anything hostile up there?",
    ],
    "ranking": [
        "Which sensor is the most alarmed right now?",
        "Rank the sensors by threat confidence.",
    ],
    "tasking": [
        "How severe is this — do we need to respond?",
        "Should I dispatch a team, and how urgently?",
    ],
}

SYSTEM_PROMPT = (
    "You are an expert at writing realistic operator queries for an airport "
    "counter-drone threat-detection system. You write the QUESTIONS an operator "
    "would ask — you never answer them and you never invent sensor readings. "
    "Every query you write must require a free-text answer (no pure yes/no that "
    "needs no explanation), must be answerable from one sensor snapshot, and must "
    "match the voice of the specified persona."
)

N_QUERIES_PER_INSTRUCTION = 8


def build_instruction(persona_key, task_key, modality_scope):
    """Assemble one instruction (user-turn) meta-prompt from the f-string parts."""
    persona_desc = PERSONAS[persona_key]
    spec = TASKS[task_key]

    # Resolve which placeholders are actually live for this variant.
    live = [p for p in spec["placeholders"]
            if p != "{modality_phrase}" or modality_scope is not None]
    ph_lines = "\n".join(f"    {p}  -> {PLACEHOLDERS[p]}" for p in live)
    ph_block = (f"Embed these placeholders VERBATIM (we substitute them later):\n{ph_lines}"
                if live else "Do NOT use any placeholders; write fully concrete questions.")

    # Modality framing (only when this task fans out over modalities).
    if modality_scope is not None:
        phr = ", ".join(f'"{p}"' for p in MODALITY_PHRASES[modality_scope])
        mod_line = (f"Focus on the {modality_scope.upper()} detector. Where {{modality_phrase}} "
                    f"appears, it will be filled with phrasings like {phr}.")
    else:
        mod_line = ""

    examples = "\n".join(f"    - {e}" for e in STYLE_EXAMPLES[task_key])

    instruction = f"""{WORLD_MODEL}

PERSONA: Write as {persona_desc}.

TASK: Produce {N_QUERIES_PER_INSTRUCTION} distinct {task_key.replace('_', ' ')} queries:
{spec['directive']}
{mod_line}

{ph_block}

RULES:
- Vary sentence shape, length, and vocabulary across the {N_QUERIES_PER_INSTRUCTION} questions.
- Stay in the persona's voice.
- Never state or guess actual readings, counts, or confidence values.
- Each question must call for a written answer, not a bare yes/no.
- Use the placeholder tokens exactly as written above, if any.

STYLE EXAMPLES (do not copy — match the register and placeholder usage only):
{examples}

OUTPUT: a JSON array of exactly {N_QUERIES_PER_INSTRUCTION} strings and nothing else."""
    return instruction


def build_dataset():
    rows = []
    rid = 0
    for persona_key, (task_key, spec) in itertools.product(PERSONAS, TASKS.items()):
        for scope in spec["modality_scopes"]:
            rows.append({
                "id": rid,
                "persona": persona_key,
                "task": task_key,
                "modality_scope": scope if scope is not None else "none",
                "placeholders": [p for p in spec["placeholders"]
                                 if p != "{modality_phrase}" or scope is not None],
                "n_queries": N_QUERIES_PER_INSTRUCTION,
                "system_prompt": SYSTEM_PROMPT,
                "instruction": build_instruction(persona_key, task_key, scope),
            })
            rid += 1
    return Dataset.from_list(rows)


if __name__ == "__main__":
    ds = build_dataset()
    ds.save_to_disk("./query_instructions_ds")
    ds.to_json("./query_instructions.jsonl")
    print(f"Built {len(ds)} instruction prompts "
          f"({len(PERSONAS)} personas x task variants).")
    print("Columns:", ds.column_names)
    print("\nTask-variant counts (per persona):")
    counts = {}
    for t, spec in TASKS.items():
        counts[t] = len(spec["modality_scopes"])
    for t, c in counts.items():
        print(f"  {t:<20} {c} variant(s)")
    print("\n----- SAMPLE INSTRUCTION (single_sensor / audio) -----\n")
    sample = next(r for r in ds if r["task"] == "single_sensor"
                  and r["modality_scope"] == "audio")
    print(sample["instruction"])


Saving the dataset (0/1 shards):   0%|          | 0/133 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Built 133 instruction prompts (7 personas x task variants).
Columns: ['id', 'persona', 'task', 'modality_scope', 'placeholders', 'n_queries', 'system_prompt', 'instruction']

Task-variant counts (per persona):
  overall_threat       1 variant(s)
  single_sensor        4 variant(s)
  check_all            4 variant(s)
  multi_sensor_group   4 variant(s)
  multi_sensor_list    4 variant(s)
  ranking              1 variant(s)
  tasking              1 variant(s)

----- SAMPLE INSTRUCTION (single_sensor / audio) -----

SYSTEM UNDER OBSERVATION — airport counter-drone network:
- 23 fixed sensors, indexed 0-22, arranged in a ring around the airfield.
- Each sensor carries three independent detectors:
    * RF      — classifies radio emissions as FRIENDLY or THREAT.
    * Audio   — classifies sound as MAMBO drone (threat), BEBOP drone (threat),
                or BACKGROUND noise (friendly).
    * Visual  — a camera score from 0 (no drone) to 1 (drone present).
- A fusion model combines everyth

## **Pass Instructions into the Model**

In [ ]:
from datasets import load_from_disk, Dataset
import json
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ────────────────────────────────────────────────────────────────────────
# LOAD THE INSTRUCTION DATASET
# ────────────────────────────────────────────────────────────────────────
ds = load_from_disk("./query_instructions_ds")
print(f"Loaded {len(ds)} instruction prompts")
print("Columns:", ds.column_names)

# ────────────────────────────────────────────────────────────────────────
# SETUP MODEL (reuse from your existing notebook)
# ────────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
llm = LLM(
    model=MODEL_ID,
    quantization="awq_marlin",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    tensor_parallel_size=1,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()


# ────────────────────────────────────────────────────────────────────────
# STAGE 2: GENERATE QUERY TEMPLATES
# ────────────────────────────────────────────────────────────────────────
sampling_params = SamplingParams(
    temperature=0.8,   # vary it; 0.8-1.0 is good for diversity
    top_p=0.95,
    max_tokens=1024,
)

def make_chat_prompt(instruction, system_prompt):
    """Format instruction + system for Qwen."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# Batch the instructions
instructions = [
    {
        "id": row["id"],
        "instruction": row["instruction"],
        "system_prompt": row["system_prompt"],
        "persona": row["persona"],
        "task": row["task"],
        "modality_scope": row["modality_scope"],
        "placeholders": row["placeholders"],
    }
    for row in ds
]

# Format all as chat prompts
formatted_prompts = [
    make_chat_prompt(inst["instruction"], inst["system_prompt"])
    for inst in instructions
]

# Run inference in batches
batch_size = 1
print(f"Generating {len(formatted_prompts)} instruction prompts through the model (batch_size={batch_size})...")
all_outputs = []

for batch_start in range(0, len(formatted_prompts), batch_size):
    batch_end = min(batch_start + batch_size, len(formatted_prompts))
    batch_prompts = formatted_prompts[batch_start:batch_end]

    print(f"  Batch {batch_start // batch_size + 1}/{(len(formatted_prompts) + batch_size - 1) // batch_size} "
          f"(indices {batch_start}-{batch_end-1})...", end=" ", flush=True)

    batch_outputs = llm.generate(batch_prompts, sampling_params)
    all_outputs.extend(batch_outputs)

    print(f"✓")

outputs = all_outputs
print(f"✓ All {len(outputs)} outputs collected")

# ────────────────────────────────────────────────────────────────────────
# PARSE RESPONSES (the model returns JSON arrays of query templates)
# ────────────────────────────────────────────────────────────────────────
templates_by_id = {}
for i, output in enumerate(outputs):
    response_text = output.outputs[0].text.strip()
    inst_meta = instructions[i]

    # Try to parse as JSON
    try:
        templates = json.loads(response_text)
        if not isinstance(templates, list):
            templates = [templates]
    except json.JSONDecodeError:
        # Fallback: wrap in a list if parsing fails
        print(f"⚠️  ID {inst_meta['id']} ({inst_meta['persona']}/{inst_meta['task']}): "
              f"JSON parse failed, treating response as a single template.")
        templates = [response_text]

    templates_by_id[inst_meta['id']] = {
        "templates": templates,
        "persona": inst_meta["persona"],
        "task": inst_meta["task"],
        "modality_scope": inst_meta["modality_scope"],
        "placeholders": inst_meta["placeholders"],
    }

print(f"Got {len(templates_by_id)} instruction → template responses")

# ────────────────────────────────────────────────────────────────────────
# STAGE 3: EXPAND TEMPLATES INTO CONCRETE QUERIES
# ────────────────────────────────────────────────────────────────────────

def expand_templates(template, task, modality_scope, k_lists=3, max_list_len=5):
    """Turn ONE returned query template into concrete queries by filling placeholders."""
    out = []
    mod_variants = MODALITY_PHRASES.get(modality_scope, [None])

    def fill(s, **kw):
        for key, val in kw.items():
            s = s.replace("{" + key + "}", str(val))
        return s

    if "{idx}" in template:                                   # single sensor -> x23
        for i in range(23):
            for mp in mod_variants:
                out.append(fill(template, idx=i, modality_phrase=mp) if mp
                           else fill(template, idx=i))
    elif "{group}" in template:                               # named groups
        for g in GROUPS:
            for mp in mod_variants:
                out.append(fill(template, group=g, modality_phrase=mp) if mp
                           else fill(template, group=g))
    elif "{indices}" in template:                             # random index subsets
        for _ in range(k_lists):
            picks = sorted(random.sample(range(23), random.randint(2, max_list_len)))
            idx_str = ", ".join(map(str, picks))
            for mp in mod_variants:
                out.append(fill(template, indices=idx_str, modality_phrase=mp) if mp
                           else fill(template, indices=idx_str))
    elif "{modality_phrase}" in template:                     # check_all
        for mp in mod_variants:
            out.append(fill(template, modality_phrase=mp))
    else:
        out.append(template)                                  # already concrete
    return out

all_queries = []

for inst_id, data in templates_by_id.items():
    for template in data["templates"]:
        try:
            expanded = expand_templates(
                template,
                data["task"],
                data["modality_scope"],
                k_lists=3,  # how many random index subsets per template
                max_list_len=5
            )
            for query in expanded:
                all_queries.append({
                    "query": query,
                    "instruction_id": inst_id,
                    "persona": data["persona"],
                    "task": data["task"],
                    "modality_scope": data["modality_scope"],
                    "template": template,  # keep the source template for debugging
                })
        except Exception as e:
            print(f"⚠️  Expansion failed for template '{template[:50]}...': {e}")

print(f"\nExpanded to {len(all_queries)} concrete queries")

# ────────────────────────────────────────────────────────────────────────
# SAVE QUERIES AS AN HF DATASET (local + push to Hub)
# ────────────────────────────────────────────────────────────────────────
query_ds = Dataset.from_list(all_queries)

# Save locally
query_ds.save_to_disk("./queries_ds")
query_ds.to_json("./queries.jsonl")
print(f"✓ Saved locally to ./queries_ds and ./queries.jsonl")

# Push to Hugging Face Hub
HF_USERNAME = "JamesResearch1216"
HF_REPO_NAME = "threat-detection-queries"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"\nPushing to {HF_REPO_ID}...")
query_ds.push_to_hub(
    HF_REPO_ID,
    token=hf_token,  # uses your existing hf_token from the secret
    private=False,   # set to True if you want it private
)
print(f"✓ Pushed to https://huggingface.co/datasets/{HF_REPO_ID}")

# ────────────────────────────────────────────────────────────────────────
# SUMMARY
# ────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"GENERATION COMPLETE")
print(f"{'='*70}")
print(f"Total queries generated: {len(all_queries):,}")
print(f"Queries per task (approx):")
task_counts = {}
for q in all_queries:
    t = q["task"]
    task_counts[t] = task_counts.get(t, 0) + 1
for t in sorted(task_counts.keys()):
    print(f"  {t:<20} {task_counts[t]:>6,}")
print(f"\nQueries per persona (approx):")
persona_counts = {}
for q in all_queries:
    p = q["persona"]
    persona_counts[p] = persona_counts.get(p, 0) + 1
for p in sorted(persona_counts.keys()):
    print(f"  {p:<20} {persona_counts[p]:>6,}")
print(f"\n📊 Dataset at:    https://huggingface.co/datasets/{HF_REPO_ID}")
print(f"{'='*70}")

# Print a few samples
print(f"\nSample queries:")
for q in all_queries[:5]:
    print(f"  [{q['persona']:<20} | {q['task']:<18}] {q['query']}")

Loaded 133 instruction prompts
Columns: ['id', 'persona', 'task', 'modality_scope', 'placeholders', 'n_queries', 'system_prompt', 'instruction']


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 06-18 19:25:07 awq_marlin.py:89] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-18 19:25:07 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_mod

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Creating parquet from Arrow format:   0%|          | 0/23 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  336kB /  336kB            

✓ Pushed to https://huggingface.co/datasets/JamesResearch1216/threat-detection-queries

GENERATION COMPLETE
Total queries generated: 22,867
Queries per task (approx):
  check_all               571
  multi_sensor_group    5,208
  multi_sensor_list     2,016
  overall_threat           56
  ranking                  56
  single_sensor        14,904
  tasking                  56

Queries per persona (approx):
  air_traffic_controller  3,315
  airport_ops_officer   3,366
  federal_air_marshal   3,382
  first_responder       2,901
  formal_commander      3,366
  incident_commander    3,174
  security_analyst      3,363

📊 Dataset at:    https://huggingface.co/datasets/JamesResearch1216/threat-detection-queries

Sample queries:
  [formal_commander     | overall_threat    ] Report immediate status: is there a confirmed threat in our airspace?
  [formal_commander     | overall_threat    ] Is the system currently identifying any drones as hostile?
  [formal_commander     | overall_threat    ] Pro